In [ ]:
import sys
sys.path.append("..")


import re
import torch
from torchviz import make_dot
import graphviz

from model.unet_upscaler_v1 import SuperResNet

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
checkpoint = torch.load('../checkpoints/upscaler.ckpt', map_location=DEVICE)
print(checkpoint['hyper_parameters'])

model = SuperResNet(start_channels=checkpoint['hyper_parameters']['start_channels'], 
                    upscale_block_length=2, 
                    downscale_block_length=2, 
                    depth=checkpoint['hyper_parameters']['depth']) 

state_dict = checkpoint['state_dict']
new_state_dict = {
    k.replace("model.", "").replace("_orig_mod.", ""): v 
    for k, v in state_dict.items()
}

model.load_state_dict(new_state_dict, strict=False)
model.to(DEVICE)
model.eval()

In [ ]:
from torchview import draw_graph

model.cpu() 
model.eval()

x = torch.randn(1, 3, 128, 128)

graph = draw_graph(
    model, 
    input_data=x,
    depth=1,            
    expand_nested=True,  
    save_graph=True,
    filename="block_architecture",
    device='cpu'
)

graph.visual_graph.attr(rankdir='TB')

graph.visual_graph.node_attr.update({
    'fontsize': '16',
    'shape': 'box', 
    'style': 'filled',
    'height': '0.2'
})

graph.visual_graph.graph_attr.update({
    'ordering': 'in',
    'nodesep': '0.1',
    'ranksep': '0.3'  
})

In [ ]:
# Graph visual refinement

dot_code = graph.visual_graph.source

def style_specific_blocks(code, class_name, color, font_size=20):
    pattern_color = rf'(\[label=<.*?{class_name}.*?>.*?fillcolor=)([\w#"]+)'
    code = re.sub(pattern_color, rf'\1"{color}"', code, flags=re.DOTALL)
    
    pattern_font = rf'(<TD ROWSPAN="2">){class_name}(<BR/>depth:\d)?(</TD>)'
    replacement_font = rf'\1<FONT POINT-SIZE="{font_size}"><B>{class_name}</B></FONT>\3'
    code = re.sub(pattern_font, replacement_font, code)
    
    return code

def simplify_blocks(code, target_keyword, display_text, color, font_size=16):
    node_pattern = re.compile(r'(\d+)\s*\[(.*?)\]', re.DOTALL)

    def replacer(match):
        node_id = match.group(1)
        content = match.group(2)
        
        is_target = False
        if re.search(rf">[^\w<]*{target_keyword}[^\w<]*<", content, re.IGNORECASE):
            is_target = True
        elif re.search(rf"label=[\"\']?{target_keyword}[\"\']?", content, re.IGNORECASE):
            is_target = True
            
        if is_target:
            new_label = (
                f'label=<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0" CELLPADDING="10" ALIGN="CENTER">'
                f'<TR>'
                    f'<TD ALIGN="CENTER" VALIGN="MIDDLE">'
                        f'<FONT POINT-SIZE="{font_size}"><B>{display_text}</B></FONT>'
                    f'</TD>'
                f'</TR>'
                f'</TABLE>>, fillcolor="{color}", style="filled,rounded"'
            )
            return f'{node_id} [{new_label}]'
        
        return match.group(0)

    return node_pattern.sub(replacer, code)


dot_code = style_specific_blocks(dot_code, "DownscaleBlock", "#f0bd9e", 18)
dot_code = style_specific_blocks(dot_code, "UpscaleBlock", "#bcf6bc", 18)
dot_code = style_specific_blocks(dot_code, "Conv2d", "#bbbff0", 18)

dot_code = simplify_blocks(dot_code, "tanh", "Tanh", "#fff0f0", 18)
dot_code = simplify_blocks(dot_code, "add", "Sum", "#ffeeee", 18)
dot_code = simplify_blocks(dot_code, "cat", "concat channels", "#ffeeee", 18)
dot_code = simplify_blocks(dot_code, "input-tensor", "Low-res image<BR/>(3ch, 128x128)", "#e1f5fe", 22)
dot_code = simplify_blocks(dot_code, "output-tensor", "High-res image<BR/>(3ch, 256x256)", "#e1f5fe", 22)

dot_code = dot_code.replace("depth:1", "")
dot_code = dot_code.replace("depth:0", "")
dot_code = dot_code.replace("UpscaleBlock", "Upscale<BR/>block")
dot_code = dot_code.replace("DownscaleBlock", "Downscale<BR/>block")
dot_code = dot_code.replace("input:", "in:")
dot_code = dot_code.replace("output:", "out:")
dot_code = dot_code.replace("(1, ", "(")
dot_code = dot_code.replace(", )", ")").replace("(, ", "(")



clean_graph = graphviz.Source(dot_code)
clean_graph